In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn
!pip install --proxy=192.168.2.10:8080 prophet 
!pip install --proxy=192.168.2.10:8080 hyperopt 

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector
import datetime
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error 
import numpy as np
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

# Setting Up the Connector

In [ ]:
# load the credentials
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']

In [ ]:
ctx = snowflake.connector.connect(**snowflake_creds)

In [ ]:
cur = ctx.cursor()

# Load Datasets

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
df.head()

# Data Preparation

In [ ]:
portfolio       = 'Buckeye'
sub_portfolio   = 'PP'
last_input_date = '2024-12-31'

In [ ]:
portfolio_df = df[df['PORTFOLIO'] == portfolio]
portfolio_df = portfolio_df[portfolio_df['SUB_PORTFOLIO'] == sub_portfolio]
portfolio_df = portfolio_df[portfolio_df['FORECAST_TYPE'] == 'Actual']

metrics = portfolio_df['METRIC'].unique()
print('Available metrics:', metrics)

In [ ]:
dff = portfolio_df.groupby(['DATE', 'METRIC']).sum().reset_index()
dff

In [ ]:
data_df = dff.pivot(
    index='DATE',
    columns='METRIC',
    values='METRIC_VALUE'
).sort_index()

# data_df

In [ ]:
train_df = data_df[data_df.index <= datetime.date.fromisoformat(last_input_date)]
test_df = data_df[data_df.index > datetime.date.fromisoformat(last_input_date)]

train_df.head()

# Modelling

In [ ]:
def plot_results(pred_df, metric):
    plt.figure(figsize=(20,6))
    plt.title(metric)
    sns.lineplot(data=pred_df, x='DATE', y='METRIC_VALUE', hue='FORECAST_TYPE', errorbar=None,)
    plt.xlabel('DATE')
    plt.ylabel('METRIC_VALUE')
    plt.xticks(pred_df['DATE'].unique(), rotation=90)

In [ ]:
def predict_metrics(train_df, metric, regressors=None):
    """
    Fit a tuned Prophet model and return predictions in long format.

    regressors : list of metric names to use as external regressors.
                 Training values come from train_df (same period).
                 Test-period values are pulled from FORECAST_TYPE='2025 0+12'
                 in the global `df`, so no data leakage occurs.
    """
    if regressors is None:
        regressors = []

    # ── pull best params ──────────────────────────────────────────────────
    growth           = best_prophet_params.get('growth', 'logistic')
    seasonality_mode = best_prophet_params.get('seasonality_mode', 'additive')
    fourier_q        = best_prophet_params.get('fourier_quarterly', 2)
    use_semi         = best_prophet_params.get('use_semiannual', 0)
    fourier_s        = best_prophet_params.get('fourier_semiannual', 1)

    # ── build training DataFrame ──────────────────────────────────────────
    pred_df_train = (train_df[metric]
                     .reset_index()
                     .rename(columns={'DATE': 'ds', metric: 'y'}))
    pred_df_train['ds'] = pd.to_datetime(pred_df_train['ds'])
    train_max = pred_df_train['y'].max()
    train_min = pred_df_train['y'].min()

    # ── build test DataFrame ──────────────────────────────────────────────
    test_input = test_df.reset_index().rename(columns={'DATE': 'ds'}).copy()
    test_input['ds'] = pd.to_datetime(test_input['ds'])

    # ── cap / floor only for logistic growth ──────────────────────────────
    if growth == 'logistic':
        pred_df_train['floor'] = max(0.0, 0.75 * train_min)
        pred_df_train['cap']   = 1.2 * train_max
        test_input['floor']    = max(0.0, 0.75 * train_min)
        test_input['cap']      = 1.2 * train_max

    # ── attach regressor columns ──────────────────────────────────────────
    for reg in regressors:
        # training values from same training slice
        reg_tr = (train_df[reg]
                  .reset_index()
                  .rename(columns={'DATE': 'ds', reg: reg}))
        reg_tr['ds'] = pd.to_datetime(reg_tr['ds'])
        pred_df_train = pred_df_train.merge(reg_tr, on='ds', how='left')

        # test-period values from the '2025 0+12' reforecast (no leakage)
        reg_test = df[
            (df['PORTFOLIO'] == portfolio) &
            (df['METRIC']    == reg) &
            (df['FORECAST_TYPE'] == '2025 0+12')
        ][['DATE', 'METRIC_VALUE']].copy()
        reg_test['DATE'] = pd.to_datetime(reg_test['DATE'])
        reg_test = reg_test.rename(columns={'DATE': 'ds', 'METRIC_VALUE': reg})
        test_input = test_input.merge(reg_test, on='ds', how='left')

    # ── build model ───────────────────────────────────────────────────────
    model = Prophet(
        growth                  = growth,
        changepoint_prior_scale = best_prophet_params['changepoint_prior_scale'],
        seasonality_prior_scale = best_prophet_params['seasonality_prior_scale'],
        changepoint_range       = best_prophet_params['changepoint_range'],
        n_changepoints          = best_prophet_params['n_changepoints'],
        seasonality_mode        = seasonality_mode,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        yearly_seasonality      = False,
    )
    # Custom seasonality components (these are what make seasonality_prior_scale work)
    model.add_seasonality(name='quarterly', period=3, fourier_order=fourier_q)
    if use_semi:
        model.add_seasonality(name='semiannual', period=6, fourier_order=fourier_s)

    for reg in regressors:
        model.add_regressor(reg)

    model.fit(pred_df_train)

    # ── predict ───────────────────────────────────────────────────────────
    forecast = model.predict(test_input)[['ds', 'yhat']].rename(
        columns={'ds': 'DATE', 'yhat': 'PRED'})
    forecast['DATE'] = forecast['DATE'].dt.date
    forecast = forecast.set_index('DATE')

    pred_df = forecast.merge(
        test_df.loc[test_df.index.isin(forecast.index), metric],
        left_index=True, right_index=True, how='inner'
    ).rename(columns={metric: 'Actual'})

    pred_df = (pred_df
               .reset_index()
               .melt(id_vars='DATE', value_vars=['Actual', 'PRED'])
               .rename(columns={'variable': 'FORECAST_TYPE', 'value': 'METRIC_VALUE'}))
    return pred_df


In [ ]:
def add_reforecast_data(pred_df, main_df, portfolio, metric, forecast_types):
    df = main_df[
        (main_df['PORTFOLIO'] == portfolio) &
        (main_df['DATE'].astype(str).isin(pred_df['DATE'].astype(str))) &
        (main_df['METRIC'] == metric) &
        (main_df['FORECAST_TYPE'].isin(forecast_types))   # ← was truncated in PDF
    ]
    df = df[pred_df.columns]
    df['DATE'] = df['DATE'].apply(
        lambda x: datetime.date.fromisoformat(str(x)))
    pred_df = pd.concat([pred_df, df], axis=0)
    return pred_df

In [ ]:
def evaluate(df_check, start, end, metric, reforecast_cols=['2025 9+3'], portfolio='PFCP', original_col='Original'):
    df_check = df_check[df_check['METRIC'] == metric]
    df_check = df_check[df_check['FORECAST_TYPE'].isin([original_col, 'Actual']+reforecast_cols)]

    start_date = datetime.date.fromisoformat(start)
    end_date = datetime.date.fromisoformat(end)
    df_check = df_check[(df_check['DATE'] >= start_date) & (df_check['DATE'] <= end_date)]

    y_true = df_check[df_check['FORECAST_TYPE'] == 'Actual'].sort_values('DATE').set_index('DATE')['METRIC_VALUE']

    result_df = pd.DataFrame(index=y_true.index, columns=['Actual'])
    result_df['Actual'] = y_true.values
    # ~~~~
    for col in [original_col] + reforecast_cols:
        y_pred = df_check[df_check['FORECAST_TYPE'] == col].sort_values('DATE').set_index('DATE')['METRIC_VALUE']
        y_pred.name = col
        result_df = result_df.merge(y_pred, how='left', left_index=True, right_index=True)

    return result_df

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Walk-forward CV splits (no data leakage) ─────────────────────────────
def ts_cv_splits(series, n_splits=3, val_size=2, min_train=10):
    n = len(series)
    for fold in range(n_splits):
        val_end   = n - (n_splits - 1 - fold) * val_size
        val_start = val_end - val_size
        if val_start < min_train:
            continue
        yield list(range(val_start)), list(range(val_start, val_end))


# ── Hyperopt search space ─────────────────────────────────────────────────
# Key fixes vs baseline:
#   1. seasonality_mode + growth are now tunable (were hardcoded → phantom dims)
#   2. Custom quarterly + optional semi-annual seasonality added (model.add_seasonality)
#      so seasonality_prior_scale actually does something
#   3. n_changepoints capped at 10 (baseline had 30 for 19 data points → pure overfit)
#   4. changepoint_prior_scale upper bound 1.0 (baseline used 1.0 fixed – too flexible)
#   5. seasonality_prior_scale upper bound raised to 20 to allow stronger seasonality

prophet_space = {
    'changepoint_prior_scale' : hp.loguniform('changepoint_prior_scale',
                                               np.log(0.001), np.log(1.0)),
    'seasonality_prior_scale' : hp.loguniform('seasonality_prior_scale',
                                               np.log(0.01), np.log(20.0)),
    'changepoint_range'       : hp.uniform('changepoint_range', 0.50, 0.90),
    'n_changepoints'          : hp.quniform('n_changepoints', 2, 10, 1),
    'seasonality_mode'        : hp.choice('seasonality_mode',
                                          ['additive', 'multiplicative']),
    'growth'                  : hp.choice('growth', ['linear', 'logistic']),
    # Custom seasonalities for monthly portfolio data
    'fourier_quarterly'       : hp.quniform('fourier_quarterly', 1, 4, 1),
    'use_semiannual'          : hp.choice('use_semiannual', [0, 1]),
    'fourier_semiannual'      : hp.quniform('fourier_semiannual', 1, 3, 1),
}


# ── CV objective function ─────────────────────────────────────────────────
def prophet_cv_objective(params):
    TARGET_METRIC    = 'Beginning Outstanding Loans'
    series           = train_df[TARGET_METRIC].dropna()
    growth           = params['growth']
    seasonality_mode = params['seasonality_mode']
    fourier_q        = int(params['fourier_quarterly'])
    use_semi         = int(params['use_semiannual'])
    fourier_s        = int(params['fourier_semiannual'])
    fold_mapes       = []

    for tr_idx, val_idx in ts_cv_splits(series, n_splits=2, val_size=3, min_train=7):
        tr  = series.iloc[tr_idx]
        val = series.iloc[val_idx]

        train_max = tr.max()
        train_min = tr.min()

        df_p   = pd.DataFrame({'ds': pd.to_datetime(tr.index),  'y': tr.values})
        df_val = pd.DataFrame({'ds': pd.to_datetime(val.index)})

        # cap/floor only needed for logistic growth
        if growth == 'logistic':
            df_p['floor']   = max(0.0, 0.75 * train_min)
            df_p['cap']     = 1.2 * train_max
            df_val['floor'] = max(0.0, 0.75 * train_min)
            df_val['cap']   = 1.2 * train_max

        try:
            m = Prophet(
                growth                  = growth,
                changepoint_prior_scale = params['changepoint_prior_scale'],
                seasonality_prior_scale = params['seasonality_prior_scale'],
                changepoint_range       = params['changepoint_range'],
                n_changepoints          = int(params['n_changepoints']),
                seasonality_mode        = seasonality_mode,
                weekly_seasonality      = False,
                daily_seasonality       = False,
                yearly_seasonality      = False,
            )
            # Monthly data: period=3 means one full quarterly cycle per 3 obs
            m.add_seasonality(name='quarterly',  period=3, fourier_order=fourier_q)
            if use_semi:
                m.add_seasonality(name='semiannual', period=6, fourier_order=fourier_s)

            m.fit(df_p)
            fc   = m.predict(df_val)['yhat'].values
            fc   = np.maximum(fc, 0)
            mape = mean_absolute_percentage_error(val.values, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)   # heavy penalty for crashes

    return {'loss': np.mean(fold_mapes), 'status': STATUS_OK}


# ── Run Hyperopt (100 trials) ──────────────────────────────────────────────
print("Running Hyperopt... please wait (100 trials)")
prophet_trials = Trials()

best_raw = fmin(
    fn        = prophet_cv_objective,
    space     = prophet_space,
    algo      = tpe.suggest,
    max_evals = 100,
    trials    = prophet_trials,
    rstate    = np.random.default_rng(42),
    verbose   = False,
)

best_prophet_params = space_eval(prophet_space, best_raw)
best_prophet_params['n_changepoints']    = int(best_prophet_params['n_changepoints'])
best_prophet_params['fourier_quarterly'] = int(best_prophet_params['fourier_quarterly'])
best_prophet_params['fourier_semiannual']= int(best_prophet_params['fourier_semiannual'])
best_prophet_params['use_semiannual']    = int(best_prophet_params['use_semiannual'])

print(f"\nBest CV MAPE     : {min(prophet_trials.losses()):.2f}%")
print(f"Baseline Prophet : 6.54%")
print(f"Statistical target: 3.21%")
print("\nBest params:")
for k, v in best_prophet_params.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
# ── Regressor ablation: rank all co-available metrics by CV MAPE impact ───
#
# For each candidate metric we fit Prophet (best hyperparams + that regressor)
# using walk-forward CV entirely within training data.
# Test-period values are NOT used here — zero leakage.
# The best regressor(s) are stored in `best_regressors` for use in predict_metrics.

TARGET_METRIC       = 'Beginning Outstanding Loans'
candidate_regressors = [m for m in train_df.columns if m != TARGET_METRIC]

print(f"Evaluating {len(candidate_regressors)} candidate regressors against '{TARGET_METRIC}'")
print(f"Columns available: {list(train_df.columns)}\n")


def cv_mape_with_regressors(reg_list):
    """Walk-forward CV MAPE using best_prophet_params + given regressors."""
    series = train_df[TARGET_METRIC].dropna()
    growth           = best_prophet_params.get('growth', 'logistic')
    seasonality_mode = best_prophet_params.get('seasonality_mode', 'additive')
    fourier_q        = best_prophet_params.get('fourier_quarterly', 2)
    use_semi         = best_prophet_params.get('use_semiannual', 0)
    fourier_s        = best_prophet_params.get('fourier_semiannual', 1)
    fold_mapes       = []

    for tr_idx, val_idx in ts_cv_splits(series, n_splits=2, val_size=3, min_train=7):
        tr  = series.iloc[tr_idx]
        val = series.iloc[val_idx]
        train_max = tr.max()
        train_min = tr.min()

        df_p   = pd.DataFrame({'ds': pd.to_datetime(tr.index), 'y': tr.values})
        df_val = pd.DataFrame({'ds': pd.to_datetime(val.index)})

        if growth == 'logistic':
            df_p['floor']   = max(0.0, 0.75 * train_min)
            df_p['cap']     = 1.2 * train_max
            df_val['floor'] = max(0.0, 0.75 * train_min)
            df_val['cap']   = 1.2 * train_max

        for reg in reg_list:
            df_p[reg]   = train_df[reg].iloc[tr_idx].values
            df_val[reg] = train_df[reg].iloc[val_idx].values

        try:
            m = Prophet(
                growth                  = growth,
                changepoint_prior_scale = best_prophet_params['changepoint_prior_scale'],
                seasonality_prior_scale = best_prophet_params['seasonality_prior_scale'],
                changepoint_range       = best_prophet_params['changepoint_range'],
                n_changepoints          = best_prophet_params['n_changepoints'],
                seasonality_mode        = seasonality_mode,
                weekly_seasonality      = False,
                daily_seasonality       = False,
                yearly_seasonality      = False,
            )
            m.add_seasonality(name='quarterly', period=3, fourier_order=fourier_q)
            if use_semi:
                m.add_seasonality(name='semiannual', period=6, fourier_order=fourier_s)
            for reg in reg_list:
                m.add_regressor(reg)
            m.fit(df_p)
            fc   = m.predict(df_val)['yhat'].values
            fc   = np.maximum(fc, 0)
            mape = mean_absolute_percentage_error(val.values, fc) * 100
            fold_mapes.append(mape)
        except Exception as e:
            fold_mapes.append(200)

    return np.mean(fold_mapes)


# ── Baseline: no regressors ───────────────────────────────────────────────
base_mape = cv_mape_with_regressors([])
print(f"{'No regressors':<45} → CV MAPE: {base_mape:.2f}%  (baseline)\n")

# ── Single-regressor sweep ────────────────────────────────────────────────
regressor_scores = {}
for reg in candidate_regressors:
    if train_df[reg].isna().any():
        print(f"  {'[SKIP] ' + reg:<43} → has NaN values")
        continue
    score = cv_mape_with_regressors([reg])
    regressor_scores[reg] = score
    delta = base_mape - score
    marker = " ✓ BETTER" if delta > 0 else ""
    print(f"  {reg:<43} → CV MAPE: {score:.2f}%  (Δ {delta:+.2f}%){marker}")

# ── Select regressors that improve baseline ───────────────────────────────
improving = {r: s for r, s in regressor_scores.items() if s < base_mape}
improving_sorted = sorted(improving.items(), key=lambda x: x[1])

if improving_sorted:
    best_regressors = [improving_sorted[0][0]]   # start with single best
    print(f"\nBest single regressor : '{best_regressors[0]}'  CV MAPE: {improving_sorted[0][1]:.2f}%")

    # ── Greedy forward add: try adding the 2nd-best regressor on top ──────
    if len(improving_sorted) > 1:
        candidate_second = [r for r, _ in improving_sorted[1:]]
        for r2 in candidate_second:
            combo_mape = cv_mape_with_regressors(best_regressors + [r2])
            print(f"  Adding '{r2}' on top → CV MAPE: {combo_mape:.2f}%")
            if combo_mape < improving_sorted[0][1]:
                best_regressors.append(r2)
                print(f"  → Added! Using regressors: {best_regressors}")
                break
else:
    best_regressors = []
    print("\nNo single regressor improved CV MAPE — using univariate model.")

print(f"\nFinal regressors chosen: {best_regressors}")


In [ ]:
TARGET_METRIC = 'Beginning Outstanding Loans'

for metric in metrics:
    if metric != TARGET_METRIC:
        continue

    # ── Predict with tuned model + best regressors ────────────────────────
    pred_df = predict_metrics(train_df, metric, regressors=best_regressors)
    pred_df = add_reforecast_data(
        pred_df        = pred_df,
        main_df        = df,
        portfolio      = portfolio,
        metric         = metric,
        forecast_types = ['Actual', 'Original', '2025 0+12'],
    )
    pred_df['DATE'] = pred_df['DATE'].apply(
        lambda x: x.date() if isinstance(x, pd.Timestamp) else x)

    # ── MAPE comparison table ─────────────────────────────────────────────
    y_true     = (pred_df[pred_df['FORECAST_TYPE'] == 'Actual']
                  .sort_values('DATE').set_index('DATE')['METRIC_VALUE'])
    result_df  = pd.DataFrame({'Actual': y_true.values}, index=y_true.index)

    for col in ['Original', '2025 0+12', 'PRED']:
        y_pred = (pred_df[pred_df['FORECAST_TYPE'] == col]
                  .sort_values('DATE').set_index('DATE')['METRIC_VALUE'])
        y_pred.name = col
        result_df = result_df.merge(y_pred, how='left',
                                    left_index=True, right_index=True)

    print(f"\n{'='*50}")
    print(f"MAPE comparison — {metric}")
    print(f"{'='*50}")
    for col in result_df.columns:
        if col == 'Actual':
            continue
        mask = result_df[col].notna()
        mape = mean_absolute_percentage_error(
            y_true=result_df['Actual'][mask],
            y_pred=result_df[col][mask]) * 100
        print(f"  {col:<20} {mape:.2f}%")

    # ── Line chart ────────────────────────────────────────────────────────
    plot_results(pred_df, metric)
    plt.show()

    # ── Component plot: verify seasonality is learned (not flat) ──────────
    print("\nPlotting Prophet components (check 'quarterly' panel for oscillations)...")
    growth           = best_prophet_params.get('growth', 'logistic')
    seasonality_mode = best_prophet_params.get('seasonality_mode', 'additive')
    fourier_q        = best_prophet_params.get('fourier_quarterly', 2)
    use_semi         = best_prophet_params.get('use_semiannual', 0)
    fourier_s        = best_prophet_params.get('fourier_semiannual', 1)

    _train = train_df[metric].reset_index().rename(columns={'DATE': 'ds', metric: 'y'})
    _train['ds'] = pd.to_datetime(_train['ds'])
    _test  = test_df.reset_index().rename(columns={'DATE': 'ds'}).copy()
    _test['ds'] = pd.to_datetime(_test['ds'])
    if growth == 'logistic':
        _train['floor'] = max(0.0, 0.75 * _train['y'].min())
        _train['cap']   = 1.2 * _train['y'].max()
        _test['floor']  = _train['floor'].iloc[0]
        _test['cap']    = _train['cap'].iloc[0]

    _m = Prophet(
        growth=growth, changepoint_prior_scale=best_prophet_params['changepoint_prior_scale'],
        seasonality_prior_scale=best_prophet_params['seasonality_prior_scale'],
        changepoint_range=best_prophet_params['changepoint_range'],
        n_changepoints=best_prophet_params['n_changepoints'],
        seasonality_mode=seasonality_mode,
        weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=False,
    )
    _m.add_seasonality(name='quarterly', period=3, fourier_order=fourier_q)
    if use_semi:
        _m.add_seasonality(name='semiannual', period=6, fourier_order=fourier_s)
    _m.fit(_train)
    _fc = _m.predict(_test)
    _m.plot_components(_fc)
    plt.show()
    break


In [ ]:
type(pred_df.iloc[0,0])

In [ ]:
6.54-3.21